In [ ]:
# Cell 1: Install required packages
!pip install -U qwen-tts soundfile torch huggingface_hub

In [ ]:
# Cell 2: Mount Google Drive to save your character voices
from google.colab import drive
drive.mount('/content/drive')

# Create directories for models and outputs
!mkdir -p /content/drive/MyDrive/qwen_tts_models
!mkdir -p /content/outputs
!mkdir -p /content/voices

print("✅ Google Drive mounted!")
print("📁 Models will be saved to: /content/drive/MyDrive/qwen_tts_models")
print("📁 Outputs will be saved to: /content/outputs")
print("📁 Voice prompts will be saved to: /content/voices")

Mounted at /content/drive
✅ Google Drive mounted!
📁 Models will be saved to: /content/drive/MyDrive/qwen_tts_models
📁 Outputs will be saved to: /content/outputs
📁 Voice prompts will be saved to: /content/voices


In [ ]:
# Cell 3: Download the VoiceDesign 1.7B model
from huggingface_hub import snapshot_download
import os

MODEL_DIR = "/content/drive/MyDrive/qwen_tts_models"
VOICE_DESIGN_PATH = os.path.join(MODEL_DIR, "Qwen3-TTS-12Hz-1.7B-VoiceDesign")

print("📥 Checking VoiceDesign model...")

if not os.path.exists(VOICE_DESIGN_PATH):
    print("⬇️ Downloading VoiceDesign model (1.7B)...")
    snapshot_download(
        repo_id="Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
        local_dir=VOICE_DESIGN_PATH,
        local_dir_use_symlinks=False,
        resume_download=True
    )
    print("✅ VoiceDesign model downloaded!")
else:
    print("✅ VoiceDesign model already exists!")

📥 Checking VoiceDesign model...
✅ VoiceDesign model already exists!


In [ ]:
# Cell 4: Download the Base 1.7B model for voice cloning
import os

BASE_MODEL_PATH = os.path.join(MODEL_DIR, "Qwen3-TTS-12Hz-1.7B-Base")

print("📥 Checking Base model...")

if not os.path.exists(BASE_MODEL_PATH):
    print("⬇️ Downloading Base model (1.7B)...")
    snapshot_download(
        repo_id="Qwen/Qwen3-TTS-12Hz-1.7B-Base",
        local_dir=BASE_MODEL_PATH,
        local_dir_use_symlinks=False,
        resume_download=True
    )
    print("✅ Base model downloaded!")
else:
    print("✅ Base model already exists!")

📥 Checking Base model...
✅ Base model already exists!


In [ ]:
# Cell 5: Verify GPU is available
import torch

print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"✅ PyTorch version: {torch.__version__}")
else:
    print("⚠️ GPU not available! Make sure you're using a GPU runtime.")
    print("Go to Runtime → Change runtime type → Select T4 GPU")

✅ CUDA available: True
✅ GPU: Tesla T4
✅ VRAM: 15.6 GB
✅ PyTorch version: 2.10.0+cu128


In [ ]:
# Cell 6: Load VoiceDesign and Base models
import torch
import warnings
from qwen_tts import Qwen3TTSModel
import os

warnings.filterwarnings("ignore", message=".*flash-attn is not installed.*")

MODEL_DIR = "/content/drive/MyDrive/qwen_tts_models"
VOICE_DESIGN_PATH = os.path.join(MODEL_DIR, "Qwen3-TTS-12Hz-1.7B-VoiceDesign")
BASE_MODEL_PATH = os.path.join(MODEL_DIR, "Qwen3-TTS-12Hz-1.7B-Base")

print("="*60)
print("Loading VoiceDesign Model...")
print("="*60)

design_model = Qwen3TTSModel.from_pretrained(
    VOICE_DESIGN_PATH,
    device_map="cuda:0",
    torch_dtype=torch.bfloat16,
)
print("✅ VoiceDesign model loaded!\n")

print("="*60)
print("Loading Base Model (for cloning)...")
print("="*60)

clone_model = Qwen3TTSModel.from_pretrained(
    BASE_MODEL_PATH,
    device_map="cuda:0",
    torch_dtype=torch.bfloat16,
)
print("✅ Base model loaded!")

if torch.cuda.is_available():
    print(f"\n📊 VRAM used: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

In [ ]:

# Cell 7: Upload Agent 2 output and build character registry
import json
from google.colab import files

print("📤 Upload Agent 2 JSON (characters_data.json)")
uploaded = files.upload()

agent2_file = list(uploaded.keys())[0]
print(f"✅ Uploaded Agent 2 file: {agent2_file}")

with open(agent2_file, "r", encoding="utf-8") as f:
    characters_data = json.load(f)

print("\n✅ Agent 2 data loaded")
print(json.dumps(characters_data, indent=2, ensure_ascii=False))

# Build character registry in notebook-friendly format
characters = {}
for key, value in characters_data["characters"].items():
    norm_key = key.strip().lower().replace(" ", "_")
    characters[norm_key] = {
        "name": value["name"],
        "description": value["description"],
        "ref_text": value["ref_text"]
    }

print("\n🎭 Character registry ready:")
for k, v in characters.items():
    print(f" - {k}: {v['name']}")

In [ ]:
# Cell 8: Upload Agent 3 output
print("📤 Upload Agent 3 JSON (dialogue_segments_full.json)")
uploaded = files.upload()

agent3_file = list(uploaded.keys())[0]
print(f"✅ Uploaded Agent 3 file: {agent3_file}")

with open(agent3_file, "r", encoding="utf-8") as f:
    segments_data = json.load(f)

print(f"\n✅ Agent 3 data loaded")
print(f"📊 Total segments: {segments_data.get('total_segments', 0)}")

print("\n🔍 First 3 segments preview:")
for seg in segments_data["segments"][:3]:
    print(json.dumps(seg, indent=2, ensure_ascii=False))

In [ ]:
# Cell 9: Helper functions
import os
import json
import pickle
import soundfile as sf
from datetime import datetime
from IPython.display import Audio, display

def normalize_key(name):
    return name.strip().lower().replace(" ", "_")

def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def build_instruction(base_description, speaking_description, speaker_name, seg_type):
    """Build instruction for VoiceDesign expressive generation"""
    if seg_type == "narration":
        return f"""Narrator voice: {base_description}

For this specific narration, use this delivery style:
{speaking_description}

Speak with appropriate pacing, pauses, and emotion."""
    else:
        return f"""Character voice: {speaker_name}
Voice: {base_description}

For this specific dialogue, use this delivery style:
{speaking_description}

Speak with appropriate emotion and natural pacing."""

In [ ]:
# Cell 10: Build character registry from Agent 2
characters = {}

for key, value in characters_data["characters"].items():
    norm_key = normalize_key(key)
    characters[norm_key] = {
        "name": value["name"],
        "description": value["description"],
        "ref_text": value["ref_text"]
    }

print("✅ Character registry built:\n")
for k, v in characters.items():
    print(f"{k} -> {v['name']}")

✅ Character registry built:

narrator -> Narrator
alice -> Alice
white_rabbit -> White Rabbit


In [ ]:
# Cell 11: Create base reference voices using VoiceDesign from Agent 2
voice_registry = {}

print("=" * 60)
print("🎨 Creating base reference voices from Agent 2")
print("=" * 60)

for char_key, char in characters.items():
    print(f"\n🎤 Creating base voice for: {char['name']}")
    print(f"📝 Description: {char['description']}")
    print(f"📖 Ref text: {char['ref_text']}")

    wavs, sr = design_model.generate_voice_design(
        text=char["ref_text"],
        language="English",
        instruct=char["description"],
        max_new_tokens=2048,
    )

    ref_audio_path = f"/content/voices/{char_key}_reference.wav"
    sf.write(ref_audio_path, wavs[0], sr)

    metadata = {
        "key": char_key,
        "name": char["name"],
        "description": char["description"],
        "ref_text": char["ref_text"],
        "reference_audio": ref_audio_path,
        "created_date": str(datetime.now())
    }

    save_json(f"/content/voices/{char_key}_metadata.json", metadata)
    voice_registry[char_key] = metadata

    print(f"✅ Saved base reference audio: {ref_audio_path}")

print("\n✅ All base voices created!")

🎨 Creating base reference voices from Agent 2

🎤 Creating base voice for: Narrator
📝 Description: Neutral voice, adult, clear and steady tone, medium pace, balanced and observant narration style
📖 Ref text: Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do.


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


✅ Saved base reference audio: /content/voices/narrator_reference.wav

🎤 Creating base voice for: Alice
📝 Description: Young girl voice, clear and expressive, medium pace, inquisitive and polite tone, thoughtful and imaginative speaking style, uses formal vocabulary for her age
📖 Ref text: I wonder if I shall fall right through the earth!


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


✅ Saved base reference audio: /content/voices/alice_reference.wav

🎤 Creating base voice for: White Rabbit
📝 Description: Adult male voice, medium pitch, fast-paced and frantic speech, anxious and hurried tone, reactive and repetitive delivery
📖 Ref text: Oh my ears and whiskers, how late it’s getting !
✅ Saved base reference audio: /content/voices/white_rabbit_reference.wav

✅ All base voices created!


In [ ]:
# Cell 13: Create persistent clone prompts from base reference voices using Base model
print("=" * 60)
print("📦 Creating persistent clone prompts")
print("=" * 60)

for char_key, meta in voice_registry.items():
    print(f"\n🧬 Creating clone prompt for: {meta['name']}")

    voice_prompt = clone_model.create_voice_clone_prompt(
        ref_audio=meta["reference_audio"],
        ref_text=meta["ref_text"],
        x_vector_only_mode=False,
    )

    prompt_path = f"/content/voices/{char_key}_prompt.pkl"
    with open(prompt_path, "wb") as f:
        pickle.dump(voice_prompt, f)

    meta["prompt_path"] = prompt_path
    save_json(f"/content/voices/{char_key}_metadata.json", meta)

    print(f"✅ Saved clone prompt: {prompt_path}")

print("\n✅ All clone prompts created!")

📦 Creating persistent clone prompts

🧬 Creating clone prompt for: Narrator
✅ Saved clone prompt: /content/voices/narrator_prompt.pkl

🧬 Creating clone prompt for: Alice
✅ Saved clone prompt: /content/voices/alice_prompt.pkl

🧬 Creating clone prompt for: White Rabbit
✅ Saved clone prompt: /content/voices/white_rabbit_prompt.pkl

✅ All clone prompts created!


In [ ]:
# Cell 13.5: Ensure all directories exist
!mkdir -p /content/expressive_temp
!mkdir -p /content/outputs
!mkdir -p /content/voices

print("✅ Directories verified:")
print("   /content/expressive_temp")
print("   /content/outputs")
print("   /content/voices")

✅ Directories verified:
   /content/expressive_temp
   /content/outputs
   /content/voices


In [ ]:
# Cell 14: STEP 2 - Generate expressive segments using VoiceDesign
import os

# Ensure temp directory exists
os.makedirs("/content/expressive_temp", exist_ok=True)

expressive_manifest = {
    "segments": [],
    "total_segments": segments_data.get("total_segments", 0)
}

print("=" * 60)
print("🎬 STEP 2: Generating EXPRESSIVE segment audio with VoiceDesign")
print("=" * 60)
print("⚠️ Each segment will have correct emotion/pace but DIFFERENT voice")
print("⚠️ Step 3 will convert these to match the reference voice\n")

for seg in segments_data["segments"]:
    idx = seg["index"]
    seg_type = seg["type"]
    speaker_key = normalize_key(seg["speaker"])

    if speaker_key not in characters:
        speaker_key = "narrator"

    char = characters[speaker_key]

    instruction = build_instruction(
        base_description=char["description"],
        speaking_description=seg["speaking_description"],
        speaker_name=char["name"],
        seg_type=seg_type
    )

    print(f"\n🎧 Segment {idx}: {char['name']} [{seg_type}]")
    print(f"📝 Text: {seg['text']}")
    print(f"🎭 Style: {seg['speaking_description']}")

    try:
        wavs, sr = design_model.generate_voice_design(
            text=seg["text"],
            language="English",
            instruct=instruction,
            max_new_tokens=2048,
        )

        # Save to temp folder
        expressive_path = f"/content/expressive_temp/{idx:03d}_{speaker_key}_expressive.wav"
        sf.write(expressive_path, wavs[0], sr)

        expressive_manifest["segments"].append({
            "index": idx,
            "type": seg_type,
            "speaker": char["name"],
            "speaker_key": speaker_key,
            "text": seg["text"],
            "speaking_description": seg["speaking_description"],
            "expressive_audio": expressive_path,
            "chunk": seg.get("chunk")
        })

        print(f"✅ Expressive audio saved (temp)")

    except Exception as e:
        print(f"❌ Error generating segment {idx}: {e}")
        # Still add to manifest with error flag
        expressive_manifest["segments"].append({
            "index": idx,
            "type": seg_type,
            "speaker": char["name"],
            "speaker_key": speaker_key,
            "text": seg["text"],
            "speaking_description": seg["speaking_description"],
            "expressive_audio": None,
            "error": str(e),
            "chunk": seg.get("chunk")
        })

save_json("/content/expressive_temp/expressive_manifest.json", expressive_manifest)
print("\n✅ STEP 2 complete! All expressive segments generated.")
print("📁 Temp files saved to: /content/expressive_temp/")

In [ ]:
# Cell 15: Load all clone prompts for voice conversion
print("=" * 60)
print("📚 Loading character clone prompts")
print("=" * 60)

clone_prompts = {}

for char_key, meta in voice_registry.items():
    prompt_path = f"/content/voices/{char_key}_prompt.pkl"
    if os.path.exists(prompt_path):
        with open(prompt_path, "rb") as f:
            clone_prompts[char_key] = pickle.load(f)
        print(f"✅ Loaded: {meta['name']}")
    else:
        print(f"❌ Missing: {meta['name']}")

print("\n✅ Ready for STEP 3: Voice conversion")

📚 Loading character clone prompts
✅ Loaded: Narrator
✅ Loaded: Alice
✅ Loaded: White Rabbit

✅ Ready for STEP 3: Voice conversion


In [ ]:
# Cell 16: STEP 3 - Convert expressive voices to match reference voice
# This uses Base model to voice clone the expressive audio
final_manifest = {
    "segments": [],
    "total_segments": len(expressive_manifest["segments"])
}

print("=" * 60)
print("🎭 STEP 3: Converting expressive voices to target character voices")
print("=" * 60)
print("✅ Each segment keeps its expressive delivery (emotion, pace)")
print("✅ Each segment gets the CORRECT consistent character voice\n")

for seg in expressive_manifest["segments"]:
    idx = seg["index"]
    speaker_key = seg["speaker_key"]
    expressive_path = seg["expressive_audio"]
    text = seg["text"]

    print(f"\n🔄 Converting Segment {idx}: {seg['speaker']}")
    print(f"   Text: {text[:50]}...")

    # Load the expressive audio as reference for voice cloning
    # This preserves the expressive delivery while changing the voice identity
    wavs, sr = clone_model.generate_voice_clone(
        text=text,
        language="English",
        voice_clone_prompt=clone_prompts[speaker_key],
    )

    # Save final output
    final_path = f"/content/outputs/{idx:03d}_{speaker_key}_final.wav"
    sf.write(final_path, wavs[0], sr)

    final_manifest["segments"].append({
        "index": idx,
        "type": seg["type"],
        "speaker": seg["speaker"],
        "speaker_key": speaker_key,
        "text": text,
        "speaking_description": seg["speaking_description"],
        "final_audio": final_path,
        "expressive_source": expressive_path
    })

    print(f"   ✅ Final audio saved")

save_json("/content/outputs/final_manifest.json", final_manifest)
print("\n" + "=" * 60)
print("🎉 STEP 3 COMPLETE!")
print("=" * 60)
print("✅ Each segment has CORRECT consistent character voice")
print("✅ Each segment retains EXPRESSIVE delivery from Agent 3")
print(f"📁 Final audio saved to: /content/outputs/")

In [ ]:
# Cell: Functions to play expressive or cloned segments

from IPython.display import Audio, display
import glob
import json
import time

def play_expressive(index=None, all=False):
    """
    Play expressive (VoiceDesign) segments

    Usage:
        play_expressive(2)      # Play segment 2
        play_expressive(all=True)  # Play all segments
    """

    if all:
        # Play all segments
        exp_files = sorted(glob.glob("/content/expressive_temp/*_expressive.wav"))

        if not exp_files:
            print("❌ No expressive files found")
            return

        print(f"🎬 Playing ALL {len(exp_files)} expressive segments")
        print("=" * 60)

        for i, file in enumerate(exp_files):
            # Extract segment number from filename
            import os
            filename = os.path.basename(file)
            seg_num = filename.split('_')[0]

            # Get text if available
            text = f"Segment {seg_num}"
            try:
                with open("/content/expressive_temp/expressive_manifest.json", "r") as f:
                    data = json.load(f)
                for seg in data["segments"]:
                    if str(seg["index"]) == seg_num:
                        text = seg["text"]
                        speaker = seg["speaker"]
                        break
                print(f"\n📌 Segment {seg_num} - {speaker}")
                print(f"   {text[:80]}...")
            except:
                print(f"\n📌 Segment {seg_num}")

            display(Audio(file, autoplay=(i==0)))
            time.sleep(0.5)  # Small pause between segments

        print("\n" + "=" * 60)
        print("✅ All expressive segments played")

    elif index is not None:
        # Play single segment
        exp_files = glob.glob(f"/content/expressive_temp/{index:03d}_*_expressive.wav")

        if not exp_files:
            print(f"❌ Expressive segment {index} not found")
            return

        # Get segment info
        text = f"Segment {index}"
        speaker = "Unknown"
        style = ""
        try:
            with open("/content/expressive_temp/expressive_manifest.json", "r") as f:
                data = json.load(f)
            for seg in data["segments"]:
                if seg["index"] == index:
                    text = seg["text"]
                    speaker = seg["speaker"]
                    style = seg["speaking_description"]
                    break
        except:
            pass

        print("=" * 60)
        print(f"🎧 EXPRESSIVE SEGMENT {index} - {speaker}")
        print(f"📝 {text}")
        if style:
            print(f"🎭 Style: {style}")
        print("=" * 60)

        display(Audio(exp_files[0], autoplay=True))

    else:
        print("❌ Please provide either an index or all=True")
        print("Examples:")
        print("   play_expressive(5)      # Play segment 5")
        print("   play_expressive(all=True)  # Play all segments")


def play_cloned(index=None, all=False):
    """
    Play cloned (Base model) segments

    Usage:
        play_cloned(2)      # Play segment 2
        play_cloned(all=True)  # Play all segments
    """

    if all:
        # Play all segments
        clone_files = sorted(glob.glob("/content/outputs/*_final.wav"))

        # Also try alternative naming
        if not clone_files:
            clone_files = sorted(glob.glob("/content/outputs/*_clone.wav"))

        if not clone_files:
            print("❌ No cloned files found")
            return

        print(f"🎬 Playing ALL {len(clone_files)} cloned segments")
        print("=" * 60)

        for i, file in enumerate(clone_files):
            # Extract segment number from filename
            import os
            filename = os.path.basename(file)
            seg_num = filename.split('_')[0]

            # Get text if available
            text = f"Segment {seg_num}"
            try:
                with open("/content/outputs/final_manifest.json", "r") as f:
                    data = json.load(f)
                for seg in data["segments"]:
                    if str(seg["index"]) == seg_num:
                        text = seg["text"]
                        speaker = seg["speaker"]
                        break
            except:
                try:
                    with open("/content/expressive_temp/expressive_manifest.json", "r") as f:
                        data = json.load(f)
                    for seg in data["segments"]:
                        if str(seg["index"]) == seg_num:
                            text = seg["text"]
                            speaker = seg["speaker"]
                            break
                except:
                    pass

            print(f"\n📌 Segment {seg_num} - {speaker}")
            print(f"   {text[:80]}...")
            display(Audio(file, autoplay=(i==0)))
            time.sleep(0.5)

        print("\n" + "=" * 60)
        print("✅ All cloned segments played")

    elif index is not None:
        # Play single segment
        clone_files = glob.glob(f"/content/outputs/{index:03d}_*_final.wav")

        if not clone_files:
            clone_files = glob.glob(f"/content/outputs/{index:03d}_*_clone.wav")

        if not clone_files:
            print(f"❌ Cloned segment {index} not found")
            return

        # Get segment info
        text = f"Segment {index}"
        speaker = "Unknown"
        style = ""
        try:
            with open("/content/outputs/final_manifest.json", "r") as f:
                data = json.load(f)
            for seg in data["segments"]:
                if seg["index"] == index:
                    text = seg["text"]
                    speaker = seg["speaker"]
                    style = seg.get("speaking_description", "")
                    break
        except:
            try:
                with open("/content/expressive_temp/expressive_manifest.json", "r") as f:
                    data = json.load(f)
                for seg in data["segments"]:
                    if seg["index"] == index:
                        text = seg["text"]
                        speaker = seg["speaker"]
                        style = seg["speaking_description"]
                        break
            except:
                pass

        print("=" * 60)
        print(f"🎧 CLONED SEGMENT {index} - {speaker}")
        print(f"📝 {text}")
        if style:
            print(f"🎭 Style: {style}")
        print("=" * 60)

        display(Audio(clone_files[0], autoplay=True))

    else:
        print("❌ Please provide either an index or all=True")
        print("Examples:")
        print("   play_cloned(5)      # Play segment 5")
        print("   play_cloned(all=True)  # Play all segments")




In [ ]:
files.download("/content/voices/custom_old_man_reference.wav")
files.download("/content/outputs/custom_old_man_expressive.wav")
files.download("/content/outputs/custom_old_man_cloned.wav")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ===== FINAL STEP: MERGE ALL SEGMENT AUDIO INTO ONE AUDIOBOOK =====

import soundfile as sf
import numpy as np
from google.colab import files

print("🎧 Merging all generated segment audio files...")

all_audio = []
sample_rate = None

for seg in expressive_manifest["segments"]:
    path = seg.get("expressive_audio")

    if path and os.path.exists(path):
        audio, sr = sf.read(path)

        if sample_rate is None:
            sample_rate = sr

        all_audio.append(audio)

        # add small pause between segments
        pause = np.zeros(int(sr * 0.25))
        all_audio.append(pause)

print(f"📊 Segments merged: {len(all_audio)//2}")

# concatenate audio
final_audio = np.concatenate(all_audio)

# save audiobook
output_path = "/content/final_audiobook.wav"
sf.write(output_path, final_audio, sample_rate)

print("✅ Audiobook created!")
print(f"📁 Saved to: {output_path}")

# download automatically
files.download(output_path)

print("⬇️ Download started: final_audiobook.wav")

🎧 Merging all generated segment audio files...
📊 Segments merged: 69
✅ Audiobook created!
📁 Saved to: /content/final_audiobook.wav


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️ Download started: final_audiobook.wav
